# Arbeitspaket (AP) 4: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung, Schlüssel &amp; Datenqualität</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (cinema.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (cinema.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `cinema.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `movies`, `cinemas`, `staff`, `screenings`, `customers`, `tickets`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `cinema_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [1]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /home/claude/out/DS_HS2026_LNW_I_Examples/AP04


### Datenbankverbindung zu PostgreSQL

In [2]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

Host:       localhost
Datenbank:  postgres


### Daten nach PostgreSQL laden

In [3]:
# Tabellen aus der SQLite-Datei cinema.db nach PostgreSQL übertragen
tables = ['movies', 'cinemas', 'staff', 'screenings', 'customers', 'tickets']

sqlite_engine = create_engine('sqlite:///cinema.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

movies         15 Zeilen
cinemas         6 Zeilen
staff          30 Zeilen
screenings    300 Zeilen
customers     505 Zeilen
tickets      2000 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `cinema.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///cinema.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `cinema.db` werden nicht gewertet.*

### A1 (4 Punkte)

Ein Analyst ärgert sich über die vielen JOINs in seinen Auswertungen. Er schlägt vor, in der Tabelle `tickets` zusätzlich die Spalten `cinemaid` und `movie_title` zu speichern:

*«Dann kann ich Tickets pro Kino oder pro Film direkt aus `tickets` zählen.»*

Ist das aus Sicht der Datenmodellierung sinnvoll? Begründen Sie Ihre Antwort und nennen Sie ein **konkretes** Problem, das mit diesem Vorschlag in `cinema.db` entstehen kann.

**Ihre Antwort:**

Nein, aus Sicht der Datenmodellierung ist das nicht sinnvoll. Beide Informationen sind bereits **eindeutig über `screeningid` ableitbar**: Eine Vorstellung (`screenings`) gehört zu genau einem Kino (`cinemaid`) und zeigt genau einen Film (`movieid` → `movies.title`). Die zusätzlichen Spalten wären deshalb **Redundanz**, was der Normalisierung widerspricht (Ziel: Redundanz reduzieren, Datenintegrität verbessern).

Konkretes Problem (**Anomalie / Inkonsistenz**): Wird eine Vorstellung in ein anderes Kino verlegt (`screenings.cinemaid` geändert) oder ein Filmtitel korrigiert (z.B. «Everything Everywhere» → «Everything Everywhere All at Once»), müssten zusätzlich alle zugehörigen Zeilen in `tickets` (bis zu 2000) nachgeführt werden. Wird das vergessen, widersprechen sich `tickets.cinemaid` und `screenings.cinemaid` — je nach Abfrageweg erhält man unterschiedliche Ticketzahlen pro Kino.

Die bessere Lösung für den Analysten ist eine **View** (oder ein Data-Warehouse-/Star-Schema für Auswertungen), die die JOINs einmal kapselt.

> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Bewertung: Entscheidend sind (1) «ableitbar über `screeningid` → Redundanz» und (2) ein konkretes Problem (Update-Anomalie bzw. Inkonsistenz). Die View ist ein Zusatz.*
>
> *Bezug: Folien Woche 1, S. 25 (Normalization: «reduce redundancy and improve data integrity»; Data Integrity Constraints), S. 29 (Star- und Snowflake-Schema für Auswertungen).*

### A2 (4 Punkte)

Bei einer Stichprobe in `cinema.db` fällt folgendes Ticket auf:

```
ticketid   purchase_datetime     screeningid   screening_datetime    (aus screenings)
T00003     2025-06-17 01:17:04   ...           2024-09-14 13:00:00
```

Beschreiben Sie das konkrete Datenproblem und nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (accurate, consistent, complete, reliable). Wie würden Sie vorgehen, um herauszufinden, ob es sich um einen Einzelfall handelt?

**Ihre Antwort:**

Das Ticket wurde laut `purchase_datetime` **rund 9 Monate nach** der Vorstellung gekauft. Ein Ticket kann aber nicht nach dem Vorstellungsbeginn gekauft werden — mindestens einer der beiden Zeitstempel ist falsch. Die Daten sind deshalb nicht **accurate** (sie bilden die Realität nicht korrekt ab); da sich zwei Tabellen widersprechen, kann man auch von fehlender **Konsistenz** (consistent) zwischen `tickets` und `screenings` sprechen.

Vorgehen: Mit einer SQL-Abfrage (JOIN `tickets` ↔ `screenings`, `WHERE t.purchase_datetime > s.screening_datetime`) zählen, wie viele Tickets betroffen sind (vgl. C1). Bei vielen Fällen handelt es sich um ein systematisches Problem (z.B. falscher Export, Zeitzonen, Testdaten) — dann ist `purchase_datetime` für Auswertungen (z.B. Vorverkaufsdauer) nicht **reliable**.

> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Bewertung: «accurate» und «consistent» gelten beide als korrekt, wenn die Begründung stimmt (Widerspruch zwischen den Zeitstempeln). Entscheidend ist die Prüfregel «Kauf nach Vorstellung = unmöglich» und der Vorschlag, alle Fälle per SQL zu zählen.*
>
> *Bezug: Folien Woche 1, S. 43 (Data Quality Management: accurate, consistent, complete, reliable; Data Profiling, Data Rules).*

### A3 (4 Punkte)

Ein Kollege möchte die Spalte `ticketid` entfernen und stattdessen `seat_number` (z.B. `A1`, `C12`) als **Primärschlüssel** der Tabelle `tickets` verwenden:

*«Jeder Sitzplatz hat eine eindeutige Nummer — das reicht als Schlüssel.»*

Ist `seat_number` als Primärschlüssel geeignet? Begründen Sie mit den Daten in `cinema.db` und schlagen Sie eine Alternative vor, die ohne `ticketid` auskommen würde.

**Ihre Antwort:**

Nein. Ein Primärschlüssel muss jede Zeile **eindeutig** identifizieren. Dieselbe Sitzplatznummer kommt aber in jeder Vorstellung (und in jedem Kino) wieder vor: In `cinema.db` gibt es 2000 Tickets, aber nur **200 verschiedene** Werte in `seat_number` — die Eindeutigkeit wäre verletzt, ein zweites Ticket für Platz `A1` könnte gar nicht gespeichert werden.

Alternative: ein **zusammengesetzter Schlüssel** aus (`screeningid`, `seat_number`) — ein Sitzplatz kann pro Vorstellung nur einmal verkauft werden. In `cinema.db` ist diese Kombination für alle 2000 Tickets eindeutig. (In der Praxis behält man trotzdem oft einen künstlichen Schlüssel wie `ticketid` und setzt auf (`screeningid`, `seat_number`) zusätzlich einen `UNIQUE`-Constraint.)

> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Kontrolle: `SELECT COUNT(*), COUNT(DISTINCT seat_number) FROM tickets` → 2000 / 200.*
>
> *Bezug: Folien Woche 1, S. 25 (Data Integrity Constraints: primary keys, foreign keys, unique constraints).*

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `cinema.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl verkaufter Tickets (`ticket_count`) und **tatsächlich erzielter Umsatz** (`revenue`, Summe der bezahlten Preise in CHF, gerundet auf 2 Dezimalstellen) pro Kino, absteigend sortiert nach `revenue`. Kinos ohne Tickets müssen nicht erscheinen.

Die SQL-Abfrage wurde aus einem bestehenden Personal-Report kopiert und angepasst:

```sql
SELECT c.name,
       COUNT(*) AS ticket_count,
       ROUND(CAST(SUM(s.base_price) AS NUMERIC), 2) AS revenue
FROM cinemas c
JOIN screenings s ON c.cinemaid    = s.cinemaid
JOIN tickets t    ON s.screeningid = t.screeningid
JOIN staff st     ON c.cinemaid    = st.cinemaid
GROUP BY c.name
ORDER BY revenue DESC;
```

Tatsächliches Ergebnis der SQL-Abfrage:

```
                name  ticket_count  revenue
0              Kinok          3969  67171.5
1          Cineplexx          3064  49948.0
2  Arthouse Metropol          2832  47780.0
3           Rex Kino          1544  25010.0
4      Pathé Küchlin           436   7181.5
```

Die Tabelle `tickets` enthält insgesamt **2000 Tickets**. Die Controllerin wundert sich: *«Allein Kinok soll fast 4000 Tickets verkauft haben?»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage und erklären Sie, warum ausgerechnet die Ticketzahl von «Pathé Küchlin» bereits stimmt.

**Begründung der Änderungen:**

1. **`JOIN staff` entfernen:** Ein Kino hat mehrere Mitarbeitende. Der JOIN auf `staff` vervielfacht deshalb jede Ticketzeile mit der Anzahl Mitarbeitenden des Kinos (**Fan-out** / Zeilenvervielfachung bei einer 1:n-Beziehung). Kinok hat 9 Mitarbeitende → 441 Tickets × 9 = 3969 Zeilen. Für die Fragestellung wird `staff` gar nicht benötigt.
2. **`SUM(t.paid_price)` statt `SUM(s.base_price)`:** `base_price` ist der Listenpreis der Vorstellung, `paid_price` der pro Ticket tatsächlich bezahlte Betrag (Rabatte). Für den *tatsächlich erzielten* Umsatz muss `paid_price` summiert werden.
3. (Optional) `COUNT(t.ticketid)` statt `COUNT(*)` macht explizit, dass Tickets gezählt werden; nach Entfernen des `staff`-JOINs liefern beide dasselbe.

«Pathé Küchlin» hat genau **eine** Mitarbeiterin bzw. einen Mitarbeiter — der JOIN vervielfacht hier mit dem Faktor 1, deshalb stimmt die Ticketzahl (436) zufällig. Der Umsatz ist aber auch dort falsch (Listenpreis 7181.50 statt bezahlt 6731.89).

Ergebnis: insgesamt 2000 Tickets; Kinok hat mit **441 Tickets** und **CHF 6'975.89** den höchsten Umsatz.

**Korrigierte SQL-Abfrage:**

In [4]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
SELECT
    c.name,
    COUNT(t.ticketid) AS ticket_count,
    ROUND(CAST(SUM(t.paid_price) AS NUMERIC), 2) AS revenue
FROM cinemas c
JOIN screenings s ON c.cinemaid    = s.cinemaid
JOIN tickets t    ON s.screeningid = t.screeningid
GROUP BY c.name
ORDER BY revenue DESC
""", con=engine)

engine.dispose()
df_B1

,name,ticket_count,revenue
0,Kinok,441,6975.89
1,Pathé Küchlin,436,6731.89
2,Rex Kino,386,5858.15
3,Cineplexx,383,5855.13
4,Arthouse Metropol,354,5595.37


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Kontrolle: `df_B1['ticket_count'].sum()` ergibt 2000 — die Summe muss der Anzahl Zeilen in `tickets` entsprechen. Werden zusätzlich Angaben aus `staff` benötigt (z.B. Anzahl Mitarbeitende pro Kino), müssen sie in einer separaten Abfrage bzw. Unterabfrage aggregiert werden.*
>
> *Bezug: Folien Woche 1, S. 27 (Relationship Cardinalities: one-to-many) und S. 40 (Joins); AP01 A1 (`paid_price` vs. `base_price`).*

### B2 (8 Punkte)

**Ziel:** Für ein Treueprogramm sollen die **5 umsatzstärksten Kundinnen und Kunden** angeschrieben werden. Ausgegeben werden sollen `customerid`, `firstname`, `lastname`, die Anzahl gekaufter Tickets (`ticket_count`) und der Umsatz (`revenue`, Summe `paid_price`, gerundet auf 2 Dezimalstellen), absteigend sortiert nach `revenue`.

```sql
SELECT c.firstname,
       c.lastname,
       COUNT(t.ticketid) AS ticket_count,
       ROUND(CAST(SUM(t.paid_price) AS NUMERIC), 2) AS revenue
FROM customers c
JOIN tickets t ON c.customerid = t.customerid
GROUP BY c.firstname, c.lastname
ORDER BY revenue DESC
LIMIT 5;
```

Tatsächliches Ergebnis:

```
  firstname lastname  ticket_count  revenue
0       Lea     Wolf            33   501.81
1       Jan  Richter            32   462.60
2       Lea    Weber            24   385.02
3      Liam   Krüger            24   375.37
4     Marie    Meier            21   340.24
```

Das Marketing ist irritiert: *«Laut Kassensystem hat keine einzelne Person mehr als 12 Tickets gekauft.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Vor- und Nachname sind **kein eindeutiger Schlüssel**: In `customers` kommen viele Namenskombinationen mehrfach vor (z.B. gibt es 8 verschiedene Kundinnen namens «Lea Wolf»). `GROUP BY c.firstname, c.lastname` fasst deshalb verschiedene Personen zu einer Gruppe zusammen — ihre Tickets und Umsätze werden addiert.

1. **Nach dem Primärschlüssel gruppieren:** `GROUP BY c.customerid, c.firstname, c.lastname`. Jede Gruppe entspricht dann genau einer Person. (Die Namen müssen in PostgreSQL ebenfalls im `GROUP BY` stehen oder aggregiert werden, weil sie im `SELECT` vorkommen.)
2. **`customerid` ausgeben:** Die ID wird gemäss Aufgabenstellung benötigt und ist für das Anschreiben eindeutig — ein Name allein würde nicht verraten, welche der 8 «Lea Wolf» gemeint ist.

Ergebnis: Die umsatzstärkste Kundin ist **CU0460 Lena Meier** mit 12 Tickets und **CHF 190.97** — keine der fünf Personen aus dem ursprünglichen Ergebnis ist tatsächlich unter den Top 5.

**Korrigierte SQL-Abfrage:**

In [5]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT
    c.customerid,
    c.firstname,
    c.lastname,
    COUNT(t.ticketid) AS ticket_count,
    ROUND(CAST(SUM(t.paid_price) AS NUMERIC), 2) AS revenue
FROM customers c
JOIN tickets t ON c.customerid = t.customerid
GROUP BY c.customerid, c.firstname, c.lastname
ORDER BY revenue DESC
LIMIT 5
""", con=engine)

engine.dispose()
df_B2

,customerid,firstname,lastname,ticket_count,revenue
0,CU0460,Lena,Meier,12,190.97
1,CU0108,Jan,Hartmann,11,182.52
2,CU0421,Marie,Meier,10,175.49
3,CU0053,Paul,Braun,10,163.80
4,CU0461,Sara,Keller,10,161.15


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Kontrolle: `SELECT firstname, lastname, COUNT(*) FROM customers GROUP BY firstname, lastname HAVING COUNT(*) > 1` zeigt 155 mehrfach vorkommende Namenskombinationen.*
>
> *Bezug: Folien Woche 1, S. 25 (Primary Keys, Data Integrity Constraints) und S. 39 (SQL-Abfrage mit `GROUP BY`).*

---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `cinema.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Prüfen Sie die Plausibilitätsregel aus A2: Wie viele Tickets wurden **nach** dem Beginn der zugehörigen Vorstellung gekauft (`purchase_datetime` später als `screening_datetime`)? Geben Sie pro Kino den Namen (`name`) und die Anzahl betroffener Tickets (`invalid_tickets`) aus, absteigend sortiert nach `invalid_tickets`. Begründen Sie Ihre Filterbedingung und warum der Vergleich der beiden Zeitstempel hier korrekt funktioniert.

**Begründung der Vorgehensweise:**

`purchase_datetime` liegt in `tickets`, `screening_datetime` und `cinemaid` in `screenings`, der Kinoname in `cinemas` — deshalb `tickets → screenings → cinemas`. `WHERE t.purchase_datetime > s.screening_datetime` behält nur die unplausiblen Tickets; `GROUP BY c.name` zählt sie pro Kino.

Die Zeitstempel sind als Text im ISO-Format `JJJJ-MM-TT hh:mm:ss` gespeichert. Bei diesem Format mit fester Länge entspricht die alphabetische Reihenfolge der zeitlichen, deshalb liefert der Vergleich in SQLite und PostgreSQL das richtige Ergebnis. (Sicherer: explizit umwandeln, in PostgreSQL z.B. mit `CAST(... AS TIMESTAMP)`.)

Ergebnis: **955 von 2000 Tickets** (fast die Hälfte) verletzen die Regel — kein Einzelfall, sondern ein systematisches Problem.

**SQL-Abfrage:**

In [6]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT
    c.name,
    COUNT(t.ticketid) AS invalid_tickets
FROM tickets t
JOIN screenings s ON t.screeningid = s.screeningid
JOIN cinemas c    ON s.cinemaid    = c.cinemaid
WHERE t.purchase_datetime > s.screening_datetime
GROUP BY c.name
ORDER BY invalid_tickets DESC
""", con=engine)

engine.dispose()
df_C1

,name,invalid_tickets
0,Rex Kino,209
1,Pathé Küchlin,201
2,Arthouse Metropol,188
3,Cineplexx,183
4,Kinok,174


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Einschätzung (nicht offiziell): Nur die Gesamtzahl 955 ohne Kinonamen entspräche «fehlende Spalten» (mittlere Mängel).*
>
> *Bezug: Folien Woche 1, S. 38 (Data Types: Datum/Zeit) und S. 43 (Data Rules, Data Profiling).*

### C2 (4 Punkte)

Welche Filmgenres bringen am meisten Umsatz? Geben Sie `genre`, die Anzahl verkaufter Tickets (`ticket_count`) und den Umsatz (`revenue`, Summe `paid_price`, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`) aus, absteigend sortiert nach `revenue`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

`genre` liegt in `movies`, `paid_price` in `tickets`. Es gibt keine direkte Verbindung — der Weg führt über `screenings` (`movies.movieid → screenings.movieid`, `screenings.screeningid → tickets.screeningid`). `GROUP BY m.genre` fasst alle Tickets pro Genre zusammen, `SUM(t.paid_price)` berechnet den tatsächlich erzielten Umsatz. `INNER JOIN` genügt, weil nur Genres mit verkauften Tickets einen Umsatz haben.

Ergebnis: **Sci-Fi** führt mit 517 Tickets und CHF 7'764.75, gefolgt von Drama (CHF 7'025.04).

**SQL-Abfrage:**

In [7]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT
    m.genre,
    COUNT(t.ticketid) AS ticket_count,
    ROUND(CAST(SUM(t.paid_price) AS NUMERIC), 2) AS revenue
FROM movies m
JOIN screenings s ON m.movieid     = s.movieid
JOIN tickets t    ON s.screeningid = t.screeningid
GROUP BY m.genre
ORDER BY revenue DESC
""", con=engine)

engine.dispose()
df_C2

,genre,ticket_count,revenue
0,Sci-Fi,517,7764.75
1,Drama,455,7025.04
2,Comedy,284,4148.48
3,Thriller,233,3677.02
4,Crime,154,2513.62
5,Action,152,2355.51
6,Romance,130,2219.21
7,Animation,75,1312.80


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Bezug: Folien Woche 1, S. 39–40 (SQL-Abfrage, Joins über mehrere Tabellen).*

### C3 (4 Punkte)

Das Programmteam möchte schwach nachgefragte Filme aus dem Programm nehmen. Welche Filme haben **weniger als 120 Tickets** verkauft? Filme **ohne** verkaufte Tickets sollen ebenfalls erscheinen (mit `0`). Geben Sie `title` und `ticket_count` aus, aufsteigend sortiert nach `ticket_count`. Begründen Sie Ihre Wahl des JOIN-Typs und warum die Bedingung `< 120` nicht im `WHERE` stehen kann.

**Begründung der Vorgehensweise:**

`LEFT JOIN` ausgehend von `movies` (über `screenings` zu `tickets`) behält auch Filme ohne Vorstellungen bzw. ohne Tickets; `COUNT(t.ticketid)` zählt für sie `0` (NULL wird nicht gezählt). Die Bedingung bezieht sich auf das **Ergebnis der Aggregation** (Anzahl pro Film) und muss deshalb in `HAVING` stehen — `WHERE` filtert einzelne Zeilen **vor** dem Gruppieren, dort ist die Anzahl noch gar nicht bekannt.

Ergebnis: 4 Filme — Spirited Away (75), Joker (89), The Matrix (98) und Inception (105). In den aktuellen Daten gibt es keinen Film ohne Tickets; die Abfrage würde ihn aber korrekt mit `0` ausweisen.

**SQL-Abfrage:**

In [8]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT
    m.title,
    COUNT(t.ticketid) AS ticket_count
FROM movies m
LEFT JOIN screenings s ON m.movieid     = s.movieid
LEFT JOIN tickets t    ON s.screeningid = t.screeningid
GROUP BY m.title
HAVING COUNT(t.ticketid) < 120
ORDER BY ticket_count ASC
""", con=engine)

engine.dispose()
df_C3

,title,ticket_count
0,Spirited Away,75
1,Joker,89
2,The Matrix,98
3,Inception,105


> **Nicht Teil der Antwort** — Bezug zu Folien bzw. Übungen
>
> *Einschätzung (nicht offiziell): Ein `INNER JOIN` liefert mit den aktuellen Daten zwar dasselbe Ergebnis, erfüllt aber die Anforderung «Filme ohne Tickets mit 0» nicht (vgl. AP01 B1) — gemäss Bewertungsschema ein «falscher JOIN-Typ».*
>
> *Bezug: Folien Woche 1, S. 40 (Joins: LEFT JOIN); AP01 B1 und C3 (`COUNT(t.ticketid)`, `HAVING`).*

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [9]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.18.44-fc-v310
Datetime: 2026-11-07 10:36:01
Python Version: 3.13.18
IP Address: 127.0.0.1
-----------------------------------
